# AVPENet — Quick-Start Demo

This notebook demonstrates how to:
1. Load a pretrained AVPENet model
2. Run inference on a single audio-visual segment
3. Visualise cross-modal attention weights
4. Reproduce the key evaluation metrics from Table 2 of the paper

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from avpenet.models.avpenet import AVPENet
from avpenet.metrics import evaluate, print_results

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Initialise Model

In [ ]:
# Build model with random weights (replace with .from_pretrained() for real use)
model = AVPENet(pretrained=False).to(device).eval()

# Alternatively, load pretrained checkpoint:
# model = AVPENet.from_pretrained('outputs/checkpoints/avpenet_best.pth')

params = model.count_parameters()
print('Parameter counts:')
for k, v in params.items():
    print(f'  {k:20s}: {v:,}')

## 2. Single-Segment Inference

In [ ]:
# Create synthetic input (replace with real data in practice)
audio  = torch.randn(1, 1, 128, 300).to(device)   # mel-spectrogram
visual = torch.randn(1, 3, 224, 224).to(device)   # face image

with torch.no_grad():
    out = model(audio, visual, return_embeddings=True, return_attention=True)

print(f'Predicted pain score: {out["pain_score"].item():.2f} / 10')
print(f'Audio embedding shape:  {out["ea"].shape}')
print(f'Visual embedding shape: {out["ev"].shape}')

## 3. Visualise Cross-Modal Attention

In [ ]:
# Attention weights shape: (B, num_heads, 1, 1)
attn = out['attn_weights']
av_weights = attn['audio_to_visual'].squeeze().cpu().numpy()  # (num_heads,)
va_weights = attn['visual_to_audio'].squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Audio-to-visual attention
axes[0].bar(range(len(av_weights)), av_weights, color='steelblue')
axes[0].set_title('Audio→Visual Attention (per head)', fontsize=13)
axes[0].set_xlabel('Attention Head')
axes[0].set_ylabel('Attention Weight')
axes[0].set_xticks(range(len(av_weights)))

# Visual-to-audio attention
axes[1].bar(range(len(va_weights)), va_weights, color='darkorange')
axes[1].set_title('Visual→Audio Attention (per head)', fontsize=13)
axes[1].set_xlabel('Attention Head')
axes[1].set_ylabel('Attention Weight')
axes[1].set_xticks(range(len(va_weights)))

plt.suptitle(f'Cross-Modal Attention  |  Pain Score = {out["pain_score"].item():.2f}', fontsize=14)
plt.tight_layout()
plt.savefig('attention_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Evaluate on Synthetic Test Set

In [ ]:
# Simulate predictions vs ground-truth
# In practice, run evaluate.py on your real test set
np.random.seed(42)
true_scores  = np.random.uniform(0, 10, 650)
# Simulate model predictions with some noise
pred_scores  = true_scores + np.random.normal(0, 0.89, 650)
pred_scores  = np.clip(pred_scores, 0, 10)
age_groups   = np.array(['neonate'] * 325 + ['adult'] * 325)

results = evaluate(pred_scores, true_scores, age_groups)
print_results(results, prefix='Synthetic Demo')

## 5. Reproduce Figure 2A — Prediction vs Ground Truth Scatter

In [ ]:
from avpenet.metrics import discretise_pain
import matplotlib.patches as mpatches

class_colors = {0: 'steelblue', 1: 'darkorange', 2: 'crimson'}
class_labels = {0: 'Low (0-3)', 1: 'Moderate (4-6)', 2: 'High (7-10)'}

classes = discretise_pain(true_scores)

fig, ax = plt.subplots(figsize=(7, 6))

for cls in [0, 1, 2]:
    mask = classes == cls
    ax.scatter(
        true_scores[mask], pred_scores[mask],
        c=class_colors[cls], label=class_labels[cls],
        alpha=0.5, s=20, edgecolors='none'
    )

# Perfect prediction line
ax.plot([0, 10], [0, 10], 'k--', linewidth=1, label='Perfect prediction')

# Regression fit
m, b = np.polyfit(true_scores, pred_scores, 1)
x_fit = np.linspace(0, 10, 100)
ax.plot(x_fit, m * x_fit + b, 'r-', linewidth=2, label=f'y={m:.2f}x+{b:.2f}')

ax.set_xlabel('Ground Truth Pain Score', fontsize=12)
ax.set_ylabel('Predicted Pain Score', fontsize=12)
ax.set_title(f'Prediction vs Ground Truth  (PCC={results["pcc"]:.2f})', fontsize=13)
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('prediction_scatter.png', dpi=150, bbox_inches='tight')
plt.show()